# Ferramenta exploratória de aderência entre formação acadêmica e domínio funcional — PMAC

**Versão revisada — fluxo integrado e auditável**

## Objetivo

Este notebook organiza uma análise exploratória da correspondência entre a área de formação acadêmica registrada e o domínio funcional inferido a partir da unidade de lotação.

O fluxo está dividido em:
1. leitura e validação do arquivo;
2. limpeza e padronização;
3. análise descritiva da base;
4. classificação das formações e das unidades;
5. aplicação da matriz de aderência;
6. análise dos casos não aplicáveis e pendentes de revisão;
7. experimento de reprodução dos rótulos com TF-IDF;
8. experimento opcional com *sentence embeddings*;
9. exportação dos resultados sem identificadores pessoais.

## Limites metodológicos

- A unidade de lotação é uma aproximação do domínio funcional; não substitui a descrição real das atribuições exercidas.
- A classificação por palavras-chave depende de regras explícitas e deve ser revisada por especialistas.
- A matriz `HIGH` / `MODERATE` abaixo é uma **parametrização inicial editável**. Antes de usar os resultados em artigo, confirme cada correspondência com a matriz institucional/metodológica aprovada. Não trate os valores como verdade objetiva.
- A regra de melhor correspondência é preservada: **qualquer correspondência ALTA prevalece; se não houver, qualquer MODERADA prevalece; caso contrário, BAIXA**.
- TF-IDF e embeddings treinados com esses rótulos medem a capacidade de reproduzir a regra, não validam independentemente a adequação da alocação.
- Similaridade de cosseno não é percentual de aderência nem probabilidade.
- O notebook não avalia competência, desempenho, produtividade ou mérito individual.

In [ ]:
# Instalação no Google Colab.
# Execute esta célula uma vez por sessão. Se as bibliotecas já estiverem instaladas,
# o pip apenas verificará/atualizará os pacotes necessários.
%pip -q install pandas numpy matplotlib seaborn openpyxl scikit-learn sentence-transformers

In [ ]:
# ============================================================
# 1. IMPORTAÇÕES E CONFIGURAÇÕES GERAIS
# ============================================================

import io
import re
import unicodedata
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Reprodutibilidade: fixa o estado aleatório usado nas divisões e modelos.
SEMENTE = 42

# Classes em ordem fixa para tabelas e matrizes de confusão.
ORDEM_CLASSES = ['ALTA', 'MODERADA', 'BAIXA']

# Opções de execução.
EXECUTAR_TFIDF = True
EXECUTAR_EMBEDDINGS = True

# Modelo multilíngue. O primeiro carregamento pode baixar os pesos.
MODELO_EMBEDDING = 'paraphrase-multilingual-MiniLM-L12-v2'

# Estilo visual dos gráficos.
sns.set_theme(style='whitegrid')

print('Ambiente preparado.')

In [ ]:
# ============================================================
# 2. LEITURA DO ARQUIVO E IDENTIFICAÇÃO DAS COLUNAS
# ============================================================

# O notebook aceita CSV, XLSX e XLS.
# Para Excel, procura as colunas necessárias nas primeiras linhas de cada aba.
# Para CSV, usa a primeira linha como cabeçalho e tenta inferir o separador.

from google.colab import files

arquivos_enviados = files.upload()

extensoes_validas = ('.csv', '.xlsx', '.xls')
nomes_validos = [
    nome for nome in arquivos_enviados
    if nome.lower().endswith(extensoes_validas)
]

if len(nomes_validos) == 0:
    raise ValueError('Envie um arquivo CSV ou Excel.')

if len(nomes_validos) > 1:
    raise ValueError(
        'Foi enviado mais de um arquivo compatível. '
        'Envie apenas a base que deseja analisar para evitar ambiguidade.'
    )

ARQUIVO = nomes_validos[0]
conteudo = arquivos_enviados[ARQUIVO]

def chave_coluna(valor):
    """Normaliza um cabeçalho para facilitar a identificação de colunas."""
    valor = '' if pd.isna(valor) else str(valor)
    valor = unicodedata.normalize('NFKD', valor)
    valor = valor.encode('ascii', 'ignore').decode('ascii')
    valor = valor.upper()
    return re.sub(r'[^A-Z0-9]+', ' ', valor).strip()

def localizar_coluna(colunas, tipo):
    """Localiza os quatro campos mínimos, tolerando acentos e espaços."""
    for coluna in colunas:
        k = chave_coluna(coluna)

        if tipo == 'AREA' and 'AREA' in k and 'FORMACAO' in k:
            return coluna
        if tipo == 'UNIDADE' and k == 'UNIDADE':
            return coluna
        if tipo == 'TIPO' and 'TIPO' in k and 'UNIDADE' in k:
            return coluna
        if tipo == 'ESCOLARIDADE' and (
            'ESCOLARIDADE' in k or ('GRAU' in k and 'ESCOLAR' in k)
        ):
            return coluna

    return None

def colunas_obrigatorias_encontradas(df):
    """Retorna os cabeçalhos localizados; campos ausentes ficam como None."""
    return {
        'AREA_FORMACAO': localizar_coluna(df.columns, 'AREA'),
        'UNIDADE': localizar_coluna(df.columns, 'UNIDADE'),
        'TIPO_UNIDADE': localizar_coluna(df.columns, 'TIPO'),
        'GRAU_ESCOLARIDADE': localizar_coluna(df.columns, 'ESCOLARIDADE')
    }

base = None
aba_escolhida = None
linha_cabecalho = None

if ARQUIVO.lower().endswith('.csv'):
    erros_leitura = []

    for codificacao in ['utf-8-sig', 'cp1252', 'latin1']:
        try:
            base = pd.read_csv(
                io.BytesIO(conteudo),
                sep=None,
                engine='python',
                encoding=codificacao
            )
            aba_escolhida = 'CSV'
            linha_cabecalho = 0
            break
        except Exception as erro:
            erros_leitura.append(f'{codificacao}: {erro}')

    if base is None:
        raise ValueError(
            'Não foi possível ler o CSV. Detalhes: ' + ' | '.join(erros_leitura)
        )

else:
    xls = pd.ExcelFile(io.BytesIO(conteudo))
    for aba in xls.sheet_names:
        for linha in range(8):
            tentativa = pd.read_excel(
                xls,
                sheet_name=aba,
                header=linha
            )
            encontrados = colunas_obrigatorias_encontradas(tentativa)

            if all(encontrados.values()):
                base = tentativa.copy()
                aba_escolhida = aba
                linha_cabecalho = linha
                break

        if base is not None:
            break

if base is None:
    raise ValueError(
        'Não foram encontradas as quatro colunas obrigatórias: '
        'área de formação, unidade, tipo de unidade e grau de escolaridade. '
        'Confira os cabeçalhos do arquivo.'
    )

# Remove colunas vazias do tipo "Unnamed" e linhas completamente vazias.
base.columns = [str(c).strip() for c in base.columns]
base = base.loc[
    :, [not chave_coluna(c).startswith('UNNAMED') for c in base.columns]
].dropna(how='all').copy()

mapeamento_colunas = colunas_obrigatorias_encontradas(base)
if not all(mapeamento_colunas.values()):
    raise ValueError(f'Falha ao mapear colunas obrigatórias: {mapeamento_colunas}')

# Renomeia os campos mínimos para nomes padronizados.
base = base.rename(
    columns={origem: destino for destino, origem in mapeamento_colunas.items()}
)

print('Arquivo:', ARQUIVO)
print('Aba:', aba_escolhida)
print('Linha de cabeçalho (base zero):', linha_cabecalho)
print('Dimensões:', base.shape)
print('Campos analíticos:', list(mapeamento_colunas.keys()))

display(base[['AREA_FORMACAO', 'UNIDADE', 'TIPO_UNIDADE', 'GRAU_ESCOLARIDADE']].head())

In [ ]:
# ============================================================
# 3. NORMALIZAÇÃO, PROTEÇÃO DE IDENTIFICADORES E CONTROLE DE QUALIDADE
# ============================================================

def norm(valor):
    """
    Padroniza texto para comparação:
    - trata ausências;
    - remove acentos;
    - converte para maiúsculas;
    - uniformiza o símbolo &;
    - remove caracteres não relevantes;
    - reduz espaços repetidos;
    - corrige algumas grafias conhecidas.

    A coluna original não é sobrescrita; a forma normalizada é armazenada
    em uma coluna separada para manter rastreabilidade.
    """
    valor = '' if pd.isna(valor) else str(valor)
    valor = unicodedata.normalize('NFKD', valor)
    valor = valor.encode('ascii', 'ignore').decode('ascii')
    valor = valor.upper().replace('&', ' E ')
    valor = re.sub(r'[^A-Z0-9/;,-]+', ' ', valor)
    valor = re.sub(r'\s+', ' ', valor).strip()

    # Correções pontuais. Evite substituições genéricas que alterem
    # palavras legítimas ou criem novas correspondências acidentais.
    correcoes = {
        'CIENCIAS SOCAIS': 'CIENCIAS SOCIAIS',
        'CIENCIAS BIOLOGICA': 'CIENCIAS BIOLOGICAS',
        'ANALISE E DESENVOLVIMENTO DE SISTEMA':
            'ANALISE E DESENVOLVIMENTO DE SISTEMAS',
        'BACHARELADO EM DIREIRO': 'BACHARELADO EM DIREITO',
        'ADMISTRATIVO': 'ADMINISTRATIVO',
        'ADMINISTRCAO': 'ADMINISTRACAO',
        'EMFERMAGEM': 'ENFERMAGEM'
    }

    for origem, destino in correcoes.items():
        # As fronteiras \b evitam substituir um trecho dentro de uma palavra.
        # Ex.: "CIENCIAS BIOLOGICAS" não deve virar "CIENCIAS BIOLOGICASS".
        valor = re.sub(
            rf'\b{re.escape(origem)}\b',
            destino,
            valor
        )

    return valor

def canonical_type(valor):
    """Padroniza o tipo oficial da unidade sem presumir valores desconhecidos."""
    valor = norm(valor)

    if valor in {'OPERACIONAL', 'OPERACIONAL PM', 'ATIVIDADE OPERACIONAL'}:
        return 'OPERACIONAL'

    if valor in {
        'ADMINISTRATIVO', 'ADMINISTRATIVA', 'ADM', 'ADMINISTRATIVO PM'
    }:
        return 'ADMINISTRATIVA'

    return 'REVISAR_TIPO_UNIDADE'

# Cópia analítica. O objeto base pode conter identificadores pessoais.
work = base.copy()

# Remove colunas que aparentam conter identificadores diretos.
# Não é uma garantia de anonimização: revise manualmente os cabeçalhos do arquivo.
padroes_identificadores = [
    r'\bNOME\b', r'NOME COMPLETO', r'CPF', r'RG', r'MATRICULA',
    r'IDENTIDADE', r'EMAIL', r'TELEFONE', r'CELULAR', r'ENDERECO'
]

colunas_identificadoras = [
    coluna for coluna in work.columns
    if any(re.search(padrao, chave_coluna(coluna)) for padrao in padroes_identificadores)
]

work = work.drop(columns=colunas_identificadoras, errors='ignore').copy()

# Identificador técnico sequencial, sem derivação de nome, matrícula ou CPF.
# Serve apenas para rastrear a linha durante a análise; não é anonimização formal.
work.insert(0, 'ID_REGISTRO_ANALISE', np.arange(1, len(work) + 1))

# Padroniza os campos necessários.
for coluna in ['AREA_FORMACAO', 'UNIDADE', 'TIPO_UNIDADE', 'GRAU_ESCOLARIDADE']:
    work[coluna] = work[coluna].astype('string').str.strip()

work['TIPO_UNIDADE_OFICIAL'] = work['TIPO_UNIDADE'].map(canonical_type)
work['AREA_FORMACAO_LIMPA'] = work['AREA_FORMACAO'].map(norm)
work['UNIDADE_LIMPA'] = work['UNIDADE'].map(norm)
work['ESCOLARIDADE_LIMPA'] = work['GRAU_ESCOLARIDADE'].map(norm)

# Classifica a situação de disponibilidade da formação.
work['SITUACAO_FORMACAO'] = np.select(
    [
        work['ESCOLARIDADE_LIMPA'].eq('SUPERIOR')
        & work['AREA_FORMACAO_LIMPA'].eq(''),
        work['ESCOLARIDADE_LIMPA'].isin(['ENSINO MEDIO', 'MEDIO']),
        work['ESCOLARIDADE_LIMPA'].isin(['SEM DADOS', 'NAO INFORMADO'])
        | work['ESCOLARIDADE_LIMPA'].eq('')
    ],
    [
        'SUPERIOR_SEM_AREA_INFORMADA',
        'ENSINO_MEDIO',
        'ESCOLARIDADE_SEM_DADOS'
    ],
    default='FORMACAO_INFORMADA'
)

print('Colunas identificadoras removidas da base analítica:', colunas_identificadoras)
print('Registros:', len(work))
print('\nTipo de unidade:')
display(work['TIPO_UNIDADE_OFICIAL'].value_counts(dropna=False).rename_axis('tipo').reset_index(name='n'))
print('\nSituação da formação:')
display(work['SITUACAO_FORMACAO'].value_counts(dropna=False).rename_axis('situacao').reset_index(name='n'))

# Verifique se a remoção automática identificou as colunas pessoais corretas.
# Se o arquivo tiver outros identificadores com nomes diferentes, acrescente-os
# manualmente à lista antes de exportar ou compartilhar resultados.

## 1. Análise descritiva global

Esta etapa descreve a base completa, antes de restringir os registros aos casos em que é possível comparar formação e domínio funcional. As categorias são apresentadas separadamente; portanto, os indicadores não devem ser somados como se fossem grupos mutuamente exclusivos.

In [ ]:
# ============================================================
# 4. ESTATÍSTICAS DESCRITIVAS DA BASE COMPLETA
# ============================================================

total = len(work)
if total == 0:
    raise ValueError('A base está vazia após a leitura e limpeza.')

def contar(condicao):
    """Conta registros que satisfazem uma condição booleana."""
    return int(condicao.fillna(False).sum())

resumo_global = pd.DataFrame({
    'indicador': [
        'Registros totais',
        'Unidades operacionais',
        'Unidades administrativas',
        'Tipo de unidade pendente de revisão',
        'Escolaridade superior',
        'Ensino médio',
        'Escolaridade sem dados',
        'Formação superior sem área informada',
        'Área de formação informada'
    ],
    'n': [
        total,
        contar(work['TIPO_UNIDADE_OFICIAL'].eq('OPERACIONAL')),
        contar(work['TIPO_UNIDADE_OFICIAL'].eq('ADMINISTRATIVA')),
        contar(work['TIPO_UNIDADE_OFICIAL'].eq('REVISAR_TIPO_UNIDADE')),
        contar(work['ESCOLARIDADE_LIMPA'].eq('SUPERIOR')),
        contar(work['ESCOLARIDADE_LIMPA'].isin(['ENSINO MEDIO', 'MEDIO'])),
        contar(work['SITUACAO_FORMACAO'].eq('ESCOLARIDADE_SEM_DADOS')),
        contar(work['SITUACAO_FORMACAO'].eq('SUPERIOR_SEM_AREA_INFORMADA')),
        contar(work['AREA_FORMACAO_LIMPA'].ne(''))
    ]
})
resumo_global['percentual_total'] = (100 * resumo_global['n'] / total).round(2)

display(resumo_global)

# Distribuição das áreas de formação: usa os textos normalizados para
# reduzir a fragmentação provocada por acentos, espaços e erros corrigidos.
areas_formacao = (
    work.loc[work['AREA_FORMACAO_LIMPA'].ne(''), 'AREA_FORMACAO_LIMPA']
    .value_counts()
    .head(25)
    .rename_axis('area_formacao')
    .reset_index(name='n')
)

denominador_areas = int(work['AREA_FORMACAO_LIMPA'].ne('').sum())
areas_formacao['percentual_entre_areas_informadas'] = (
    100 * areas_formacao['n'] / denominador_areas
).round(2) if denominador_areas else 0

display(areas_formacao)

# Gráfico de composição por tipo oficial de unidade.
tipo_unidade = (
    work['TIPO_UNIDADE_OFICIAL']
    .value_counts()
    .reindex(
        ['OPERACIONAL', 'ADMINISTRATIVA', 'REVISAR_TIPO_UNIDADE'],
        fill_value=0
    )
)

ax = tipo_unidade.sort_values().plot(kind='barh', figsize=(9, 4))
ax.set_title('Registros por tipo oficial de unidade')
ax.set_xlabel('Número de registros')
ax.set_ylabel('Tipo de unidade')
plt.tight_layout()
plt.show()

## 2. Classificação das formações acadêmicas

A função abaixo atribui um ou mais domínios acadêmicos a cada formação. A classificação múltipla é intencional: uma formação pode ter componentes interdisciplinares. Na etapa de aderência, será aplicada a regra de melhor correspondência, sem penalizar a formação por possuir mais de um domínio.

In [ ]:
# ============================================================
# 5. DOMÍNIOS ACADÊMICOS
# ============================================================

def contem_algum(texto, padroes):
    """Retorna True se algum padrão textual estiver presente."""
    return any(padrao in texto for padrao in padroes)

def degree_domains(formacao):
    """
    Classifica uma formação em um ou mais domínios acadêmicos.

    Retornos especiais:
    - SEM_DADOS: não há formação comparável informada;
    - NAO_CLASSIFICADA: há texto, mas as regras atuais não o reconhecem.

    A classificação é baseada em palavras-chave e deve ser auditada
    com a lista real de formações da base.
    """
    texto = norm(formacao)

    if texto in {
        '', 'SEM DADOS', 'ENSINO MEDIO', 'MEDIO',
        'NAO INFORMADO', 'NAO SE APLICA'
    }:
        return ['SEM_DADOS']

    dominios = []

    # Formação jurídica.
    if contem_algum(texto, [
        'DIREITO', 'JURID', 'NOTAR', 'CARTOR',
        'SERVICO JURIDICO', 'ASSESSORIA JURIDICA'
    ]):
        dominios.append('JURIDICA')

    # Gestão, administração, economia e áreas correlatas.
    if contem_algum(texto, [
        'ADMINISTR', 'GESTAO PUBLICA', 'GESTAO DE POLITICAS',
        'POLITICAS PUBLICAS', 'RECURSOS HUMANOS', 'RECURSO HUMANO',
        'PROCESSOS GERENCIAIS', 'GESTAO COMERCIAL', 'GESTAO FINANCEIRA',
        'CONTABIL', 'CIENCIAS ECONOMIC', 'ECONOMIA', 'LOGISTICA',
        'NEGOCIO', 'MARKETING', 'EMPREENDEDOR', 'PROJETOS',
        'COMERCIO EXTERIOR', 'GESTAO HOSPITALAR', 'GESTAO DA QUALIDADE',
        'GESTAO EM SAUDE', 'AGRONEGOCIO', 'MEDIACAO', 'SERVICO SOCIAL'
    ]):
        dominios.append('GESTAO_ADMIN')

    # Saúde.
    if contem_algum(texto, [
        'ENFERMAG', 'MEDICINA', 'ODONTO', 'FISIOTER',
        'NUTRICAO', 'RADIOLOG', 'FARMAC', 'BIOMEDIC',
        'SAUDE', 'GERONTOLOG', 'VETERINARIA'
    ]):
        dominios.append('SAUDE')

    # Humanidades, educação e comunicação.
    if contem_algum(texto, [
        'PEDAGOG', 'HISTORIA', 'PORTUG', 'VERNAC',
        'ESPANHOL', 'INGLES', 'FILOSOF', 'SOCIOLOG',
        'CIENCIAS SOCIAIS', 'MUSICA', 'ARTES', 'TEATRO',
        'JORNAL', 'COMUNICACAO SOCIAL', 'PUBLICIDADE',
        'PRODUCAO PUBLICITARIA', 'SERVICOS SOCIAIS',
        'DESIGN GRAFICO', 'GEOGRAF', 'PSICOLOG',
        'TEOLOG', 'LINGUA', 'GASTRONOMIA'
    ]):
        dominios.append('HUMANIDADES_COMUNICACAO')

    # Educação física: tratada separadamente de Física como ciência.
    if contem_algum(texto, ['EDUCACAO FISICA', 'ESPORTE', 'CIENCIAS DO ESPORTE']):
        dominios.append('EDUCACAO_FISICA')

    # Ciências exatas, engenharias e tecnologia.
    if contem_algum(texto, [
        'MATEMAT', 'QUIMICA', 'CIENCIAS BIOLOG', 'BIOLOG',
        'ENGENH', 'ARQUITET', 'SISTEMAS', 'TECNOLOGIA',
        'COMPUT', 'REDES', 'SOFTWARE', 'INTELIGENCIA ARTIFICIAL',
        'BANCO DE DADOS', 'ESTATIST', 'SEGURANCA DA INFORMACAO'
    ]):
        dominios.append('STEM_TECNOLOGIA')

    # Física como ciência, excluindo explicitamente Educação Física.
    if 'FISICA' in texto and 'EDUCACAO FISICA' not in texto:
        dominios.append('STEM_TECNOLOGIA')

    # Ciências ambientais e agropecuárias.
    if contem_algum(texto, [
        'AMBIENTAL', 'FLORESTAL', 'AGRONOM',
        'AGRICULT', 'ZOOTECNIA', 'RECURSOS NATURAIS'
    ]):
        dominios.append('AMBIENTAL_AGRO')

    # Segurança pública e áreas de segurança especializadas.
    if contem_algum(texto, [
        'SEGURANCA PUBLICA', 'SEGURANCA PRIVADA',
        'DEFESA CIBERNETICA', 'INVESTIGACAO FORENSE',
        'PERICIA CRIMINAL', 'CIENCIAS POLICIAIS',
        'SEGURANCA DO TRABALHO', 'GESTAO DE SEGURANCA',
        'GESTAO EM SEGURANCA', 'SEGURANCA NO TRANSITO',
        'GESTAO DE TRANSITO', 'TRANSITO'
    ]):
        dominios.append('SEGURANCA')

    return sorted(set(dominios)) if dominios else ['NAO_CLASSIFICADA']

work['DOMINIOS_FORMACAO'] = work['AREA_FORMACAO_LIMPA'].apply(degree_domains)

# Relatório de formações que exigem revisão manual.
revisar_formacoes = work.loc[
    work['DOMINIOS_FORMACAO'].apply(lambda x: 'NAO_CLASSIFICADA' in x),
    ['ID_REGISTRO_ANALISE', 'AREA_FORMACAO', 'AREA_FORMACAO_LIMPA']
].copy()

print('Formações não classificadas pelas regras atuais:', len(revisar_formacoes))
display(revisar_formacoes.head(50))

# Frequência de domínios acadêmicos. Uma linha pode contribuir para mais
# de um domínio, por isso a soma pode superar o número de pessoas.
frequencia_dominios = (
    work[['ID_REGISTRO_ANALISE', 'DOMINIOS_FORMACAO']]
    .explode('DOMINIOS_FORMACAO')
    ['DOMINIOS_FORMACAO']
    .value_counts()
    .rename_axis('dominio_academico')
    .reset_index(name='n')
)
display(frequencia_dominios)

## 3. Inferência do domínio funcional da unidade

A função `unit_domain` classifica o tipo e o domínio funcional com base no nome normalizado da unidade. As regras mais específicas aparecem antes das mais gerais para reduzir colisões de palavras-chave.

**Atenção:** nas unidades operacionais, o domínio `POLICIAMENTO_TERRITORIAL` funciona como categoria residual apenas quando essa hipótese é aceita para a unidade. Nas administrativas, unidade não reconhecida permanece pendente de revisão, em vez de receber automaticamente `GESTAO_ADMIN`.

In [ ]:
# ============================================================
# 6. DOMÍNIOS FUNCIONAIS E CLASSIFICAÇÃO DAS UNIDADES
# ============================================================

def unit_domain(unidade, tipo_unidade):
    """
    Retorna (tipo_unidade_padronizado, dominio_funcional).

    Unidades administrativas não reconhecidas são marcadas para revisão.
    Unidades operacionais sem especialização reconhecida recebem
    POLICIAMENTO_TERRITORIAL como categoria residual provisória.
    """
    u = norm(unidade)
    tipo = canonical_type(tipo_unidade)

    if not u:
        return tipo, 'UNIDADE_SEM_DADOS'

    if tipo == 'REVISAR_TIPO_UNIDADE':
        return tipo, 'REVISAR_CLASSIFICACAO'

    if tipo == 'OPERACIONAL':
        # Regras específicas primeiro.
        if contem_algum(u, ['TRANSITO', 'BPTRAN', 'BATALHAO DE TRANSITO']):
            return tipo, 'TRANSITO'
        if contem_algum(u, ['AMBIENTAL', 'BPA', 'POLICIAMENTO AMBIENTAL']):
            return tipo, 'AMBIENTAL'
        if contem_algum(u, ['CINOTECNIA', 'CANIL', 'CAES']):
            return tipo, 'CINOTECNIA'
        if contem_algum(u, ['BOPE', 'CHOQUE', 'ROTAM', 'GIRO', 'INTERVENCAO TATICA']):
            return tipo, 'INTERVENCAO_TATICA'

        # Categoria residual. Deve ser conferida com a estrutura orgânica
        # e as atribuições efetivas da unidade.
        return tipo, 'POLICIAMENTO_TERRITORIAL'

    if tipo == 'ADMINISTRATIVA':
        # Regras específicas primeiro para evitar que palavras amplas,
        # como "gestão" ou "administração", capturem áreas especializadas.
        if contem_algum(u, ['SAUDE', 'HOSPITAL', 'ODONTO', 'JUNTA MEDICA', 'ODONTOLOG']):
            return tipo, 'SAUDE'
        if contem_algum(u, ['ENSINO', 'FORMACAO', 'INSTRUCAO', 'ACADEMIA', 'ESCOLA']):
            return tipo, 'EDUCACAO'
        if contem_algum(u, ['ASSESSORIA JURIDICA', 'JURIDIC', 'JUSTICA']):
            return tipo, 'JURIDICA'
        if contem_algum(u, ['INTELIGENCIA', 'ANALISE CRIMINAL', 'CORPINT']):
            return tipo, 'INTELIGENCIA'
        if contem_algum(u, ['TECNOLOGIA', 'INFORMATICA', 'SISTEMAS']):
            return tipo, 'TECNOLOGIA'
        if contem_algum(u, ['COMUNICACAO', 'IMPRENSA', 'PUBLICIDADE']):
            return tipo, 'COMUNICACAO'
        if contem_algum(u, ['CORREGEDORIA', 'CONTROLE INTERNO', 'OUVIDORIA', 'AUDITORIA']):
            return tipo, 'CONTROLE'
        if contem_algum(u, ['LOGISTICA', 'PATRIMONIO', 'ALMOXARIFADO', 'TRANSPORTE', 'MATERIAL']):
            return tipo, 'LOGISTICA'
        if contem_algum(u, ['PLANEJAMENTO', 'PROJETOS', 'CONVENIOS', 'ESTATISTICA']):
            return tipo, 'PLANEJAMENTO'
        if contem_algum(u, ['DRH', 'RECURSOS HUMANOS', 'PESSOAL', 'ADMINISTRACAO', 'DIRETORIA GERAL']):
            return tipo, 'GESTAO_ADMIN'

        return tipo, 'REVISAR_CLASSIFICACAO'

    return tipo, 'REVISAR_CLASSIFICACAO'

resultado_unidades = work.apply(
    lambda linha: unit_domain(
        linha['UNIDADE_LIMPA'],
        linha['TIPO_UNIDADE_OFICIAL']
    ),
    axis=1
)

work['TIPO_UNIDADE_CLASSIFICADO'] = resultado_unidades.apply(lambda x: x[0])
work['DOMINIO_UNIDADE'] = resultado_unidades.apply(lambda x: x[1])

# Inspeção de combinações únicas. Essa tabela é essencial para auditar
# regras de palavras-chave que podem classificar unidades incorretamente.
auditoria_unidades = (
    work.groupby(
        ['TIPO_UNIDADE_OFICIAL', 'UNIDADE_LIMPA', 'DOMINIO_UNIDADE'],
        dropna=False
    )
    .size()
    .reset_index(name='n')
    .sort_values(['DOMINIO_UNIDADE', 'n'], ascending=[True, False])
)

display(auditoria_unidades)

pendencias_unidade = auditoria_unidades[
    auditoria_unidades['DOMINIO_UNIDADE'].isin([
        'REVISAR_CLASSIFICACAO', 'UNIDADE_SEM_DADOS'
    ])
]
print('Combinações de unidade pendentes de revisão:', len(pendencias_unidade))
display(pendencias_unidade)

## 4. Matriz de aderência e regra de melhor correspondência

A matriz associa cada domínio funcional a domínios acadêmicos com correspondência alta ou moderada. O resultado é definido pela melhor correspondência encontrada entre os domínios acadêmicos da formação:

1. se qualquer domínio acadêmico estiver em `HIGH`, o resultado é `ALTA`;
2. caso contrário, se qualquer domínio estiver em `MODERATE`, o resultado é `MODERADA`;
3. se houver formação classificável e domínio funcional reconhecido, mas nenhuma correspondência, o resultado é `BAIXA`.

**Importante:** os conjuntos abaixo são uma proposta operacional inicial baseada nos domínios do projeto. Eles precisam ser confrontados com a matriz que será efetivamente adotada no estudo e validados por especialistas. Edite-os antes de interpretar resultados substantivos.

In [ ]:
# ============================================================
# 7. MATRIZ DE CORRESPONDÊNCIA (PARAMETRIZAÇÃO EDITÁVEL)
# ============================================================

# Cada chave é um domínio funcional da unidade.
# Os valores são domínios acadêmicos considerados de aderência ALTA.
HIGH = {
    'TRANSITO': {'SEGURANCA'},
    'AMBIENTAL': {'AMBIENTAL_AGRO'},
    'INTERVENCAO_TATICA': {'SEGURANCA', 'EDUCACAO_FISICA'},
    'CINOTECNIA': {'SAUDE', 'AMBIENTAL_AGRO', 'SEGURANCA'},
    'POLICIAMENTO_TERRITORIAL': {'SEGURANCA'},
    'SAUDE': {'SAUDE'},
    'EDUCACAO': {'HUMANIDADES_COMUNICACAO', 'EDUCACAO_FISICA'},
    'JURIDICA': {'JURIDICA'},
    'INTELIGENCIA': {'SEGURANCA', 'STEM_TECNOLOGIA'},
    'TECNOLOGIA': {'STEM_TECNOLOGIA'},
    'COMUNICACAO': {'HUMANIDADES_COMUNICACAO'},
    'CONTROLE': {'JURIDICA'},
    'LOGISTICA': {'GESTAO_ADMIN', 'STEM_TECNOLOGIA'},
    'PLANEJAMENTO': {'GESTAO_ADMIN'},
    'GESTAO_ADMIN': {'GESTAO_ADMIN'}
}

# Correspondências MODERADAS são utilizadas somente quando não há
# correspondência ALTA. Não repita aqui um par já definido como ALTO.
MODERATE = {
    'TRANSITO': {'STEM_TECNOLOGIA', 'GESTAO_ADMIN'},
    'AMBIENTAL': {'STEM_TECNOLOGIA', 'SAUDE'},
    'INTERVENCAO_TATICA': {'SAUDE', 'HUMANIDADES_COMUNICACAO'},
    'CINOTECNIA': {'EDUCACAO_FISICA', 'STEM_TECNOLOGIA'},
    'POLICIAMENTO_TERRITORIAL': {
        'JURIDICA', 'EDUCACAO_FISICA', 'HUMANIDADES_COMUNICACAO'
    },
    'SAUDE': {'GESTAO_ADMIN', 'EDUCACAO_FISICA'},
    'EDUCACAO': {'GESTAO_ADMIN', 'JURIDICA'},
    'JURIDICA': {'GESTAO_ADMIN', 'HUMANIDADES_COMUNICACAO'},
    'INTELIGENCIA': {'JURIDICA', 'HUMANIDADES_COMUNICACAO', 'GESTAO_ADMIN'},
    'TECNOLOGIA': {'GESTAO_ADMIN'},
    'COMUNICACAO': {'GESTAO_ADMIN', 'STEM_TECNOLOGIA'},
    'CONTROLE': {'GESTAO_ADMIN', 'STEM_TECNOLOGIA'},
    'LOGISTICA': {'SAUDE'},
    'PLANEJAMENTO': {'STEM_TECNOLOGIA', 'JURIDICA'},
    'GESTAO_ADMIN': {'JURIDICA', 'STEM_TECNOLOGIA'}
}

DOMINIOS_FUNCIONAIS_ESPERADOS = set(HIGH)
DOMINIOS_ACADEMICOS_VALIDOS = {
    'JURIDICA', 'GESTAO_ADMIN', 'SAUDE',
    'HUMANIDADES_COMUNICACAO', 'EDUCACAO_FISICA',
    'STEM_TECNOLOGIA', 'AMBIENTAL_AGRO', 'SEGURANCA'
}

# Verificações estruturais evitam chaves ausentes e conflitos entre níveis.
assert set(MODERATE) == DOMINIOS_FUNCIONAIS_ESPERADOS, (
    'HIGH e MODERATE precisam conter exatamente as mesmas chaves funcionais.'
)

for dominio_funcional in DOMINIOS_FUNCIONAIS_ESPERADOS:
    assert HIGH[dominio_funcional].issubset(DOMINIOS_ACADEMICOS_VALIDOS)
    assert MODERATE[dominio_funcional].issubset(DOMINIOS_ACADEMICOS_VALIDOS)
    assert HIGH[dominio_funcional].isdisjoint(MODERATE[dominio_funcional]), (
        f'Há domínios repetidos em ALTA e MODERADA para {dominio_funcional}.'
    )

def adherence(domains, dominio_funcional):
    """
    Aplica a regra de melhor correspondência.

    Retorna:
      (4, 'ALTA')      se houver ao menos uma correspondência HIGH;
      (3, 'MODERADA')  se não houver HIGH, mas houver MODERATE;
      (2, 'BAIXA')     se formação e unidade forem classificáveis,
                       mas não houver correspondência;
      (None, motivo)   se o caso não puder ser avaliado.
    """
    if dominio_funcional not in HIGH:
        return None, 'NAO_APLICAVEL_UNIDADE'

    if not domains or 'SEM_DADOS' in domains:
        return None, 'NAO_APLICAVEL_FORMACAO'

    # Formação com texto, mas sem domínio reconhecido, não deve ser
    # classificada automaticamente como BAIXA: precisa de revisão.
    dominios_validos = set(domains) & DOMINIOS_ACADEMICOS_VALIDOS
    if not dominios_validos:
        return None, 'REVISAR_FORMACAO'

    if dominios_validos & HIGH[dominio_funcional]:
        return 4, 'ALTA'

    if dominios_validos & MODERATE[dominio_funcional]:
        return 3, 'MODERADA'

    return 2, 'BAIXA'

print('Matriz estruturalmente consistente.')
print('ATENÇÃO: consistência de código não equivale a validação substantiva da matriz.')

In [ ]:
# ============================================================
# 8. APLICAÇÃO DA MATRIZ AOS REGISTROS
# ============================================================

resultados = work.apply(
    lambda linha: adherence(
        linha['DOMINIOS_FORMACAO'],
        linha['DOMINIO_UNIDADE']
    ),
    axis=1
)

work['ADERENCIA_CODIGO'] = resultados.apply(lambda x: x[0]).astype('Int64')
work['ADERENCIA_CATEGORIA'] = resultados.apply(lambda x: x[1])

# Indicador binário de baixa aderência:
# 1 = BAIXA; 0 = ALTA ou MODERADA; NA = caso não aplicável/revisão.
work['INDICADOR_BAIXA_ADERENCIA'] = pd.Series(
    pd.NA, index=work.index, dtype='Int64'
)
work.loc[
    work['ADERENCIA_CATEGORIA'].isin(['ALTA', 'MODERADA']),
    'INDICADOR_BAIXA_ADERENCIA'
] = 0
work.loc[
    work['ADERENCIA_CATEGORIA'].eq('BAIXA'),
    'INDICADOR_BAIXA_ADERENCIA'
] = 1

# A base aplicável contém somente registros com categoria ALTA, MODERADA
# ou BAIXA. Casos pendentes ficam fora do denominador de aderência.
aplicaveis = work[
    work['ADERENCIA_CATEGORIA'].isin(ORDEM_CLASSES)
].copy()

nao_aplicaveis_revisao = work[
    ~work['ADERENCIA_CATEGORIA'].isin(ORDEM_CLASSES)
].copy()

print('Base total:', len(work))
print('Base aplicável:', len(aplicaveis))
print('Não aplicáveis ou pendentes:', len(nao_aplicaveis_revisao))

distribuicao_aderencia = (
    aplicaveis['ADERENCIA_CATEGORIA']
    .value_counts()
    .reindex(ORDEM_CLASSES, fill_value=0)
    .rename_axis('aderencia')
    .reset_index(name='n')
)

denominador_aplicavel = len(aplicaveis)
distribuicao_aderencia['percentual_base_aplicavel'] = (
    100 * distribuicao_aderencia['n'] / denominador_aplicavel
).round(2) if denominador_aplicavel else 0

display(distribuicao_aderencia)

print('Motivos de não aplicabilidade/revisão:')
display(
    nao_aplicaveis_revisao['ADERENCIA_CATEGORIA']
    .value_counts(dropna=False)
    .rename_axis('motivo')
    .reset_index(name='n')
)

# Cruzamento por tipo oficial de unidade.
tabela_por_tipo = pd.crosstab(
    aplicaveis['TIPO_UNIDADE_OFICIAL'],
    aplicaveis['ADERENCIA_CATEGORIA']
).reindex(columns=ORDEM_CLASSES, fill_value=0)

display(tabela_por_tipo)

# Visualização da distribuição na base aplicável.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

distribuicao_aderencia.set_index('aderencia')['n'].plot(
    kind='bar', ax=axes[0]
)
axes[0].set_title('Aderência — base aplicável')
axes[0].set_xlabel('Categoria')
axes[0].set_ylabel('Registros')
axes[0].tick_params(axis='x', rotation=0)

tabela_por_tipo.plot(kind='bar', ax=axes[1])
axes[1].set_title('Aderência por tipo de unidade')
axes[1].set_xlabel('Tipo de unidade')
axes[1].set_ylabel('Registros')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Aderência')

plt.tight_layout()
plt.show()

## 5. Experimento 1 — TF-IDF + regressão logística

Este experimento avalia se um classificador textual consegue reproduzir as categorias geradas pela matriz. O texto de entrada inclui formação, unidade e domínio funcional. Como a categoria-alvo foi calculada por regras que usam esses mesmos elementos, o desempenho é uma medida de reprodução do procedimento, não uma validação independente.

O modelo é comparado a uma linha de base majoritária (`DummyClassifier`). A divisão é estratificada para preservar a proporção das classes. Como registros da mesma unidade podem aparecer nos dois subconjuntos, a avaliação não mede necessariamente generalização para unidades nunca vistas.

In [ ]:
# ============================================================
# 9. TF-IDF + REGRESSÃO LOGÍSTICA
# ============================================================

def perfil_funcional(dominio):
    """Descrição textual curta para representar o domínio funcional."""
    descricoes = {
        'GESTAO_ADMIN': 'gestão administrativa, recursos humanos, administração e apoio ao comando',
        'EDUCACAO': 'ensino, formação, instrução, capacitação e desenvolvimento profissional',
        'SAUDE': 'saúde, medicina, enfermagem, odontologia e atendimento clínico',
        'INTELIGENCIA': 'inteligência, análise criminal, investigação e produção de conhecimento',
        'CONTROLE': 'corregedoria, auditoria, controle interno, ouvidoria e fiscalização',
        'LOGISTICA': 'logística, patrimônio, almoxarifado, contratos e materiais',
        'PLANEJAMENTO': 'planejamento, projetos, convênios e gestão estratégica',
        'COMUNICACAO': 'comunicação social, imprensa, jornalismo e relações públicas',
        'JURIDICA': 'direito, assessoria jurídica, processos e legislação',
        'TECNOLOGIA': 'tecnologia da informação, sistemas, dados, redes e software',
        'TRANSITO': 'trânsito, fiscalização viária e policiamento de trânsito',
        'AMBIENTAL': 'policiamento ambiental, meio ambiente e recursos naturais',
        'INTERVENCAO_TATICA': 'operações especiais, intervenção tática e patrulhamento tático',
        'CINOTECNIA': 'cinotecnia, cães policiais, adestramento, busca e detecção',
        'POLICIAMENTO_TERRITORIAL': 'policiamento territorial, patrulhamento ostensivo e atendimento de ocorrências'
    }
    return descricoes.get(dominio, 'domínio funcional não descrito')

tfidf_model = None
pred_tfidf = None
baseline = None
pred_baseline = None

base_ml = aplicaveis.copy()

# Remove registros sem rótulo ou sem texto acadêmico utilizável.
base_ml = base_ml[
    base_ml['ADERENCIA_CATEGORIA'].isin(ORDEM_CLASSES)
].copy()

base_ml['AREA_FORMACAO_TEXTO'] = (
    base_ml['AREA_FORMACAO_LIMPA'].fillna('').astype(str)
)
base_ml['UNIDADE_TEXTO'] = (
    base_ml['UNIDADE_LIMPA'].fillna('').astype(str)
)

base_ml['TEXTO_MODELO'] = base_ml.apply(
    lambda linha: (
        f"formação: {linha['AREA_FORMACAO_TEXTO']}; "
        f"unidade: {linha['UNIDADE_TEXTO']}; "
        f"domínio funcional: {perfil_funcional(linha['DOMINIO_UNIDADE'])}"
    ),
    axis=1
)

X_texto = base_ml['TEXTO_MODELO'].astype(str)
y = base_ml['ADERENCIA_CATEGORIA'].astype(str)

contagens_classes = y.value_counts()
print('Distribuição das classes:')
display(
    contagens_classes.reindex(ORDEM_CLASSES, fill_value=0)
    .rename_axis('classe').reset_index(name='n')
)

if len(contagens_classes) < 2 or contagens_classes.min() < 2:
    raise ValueError(
        'Não há pelo menos dois registros em cada classe para uma divisão '
        'estratificada. Revise a matriz ou use uma estratégia de validação apropriada.'
    )

# A mesma divisão será reutilizada na avaliação de embeddings, permitindo
# comparação mais justa entre os dois métodos.
indices = np.arange(len(base_ml))
idx_treino, idx_teste = train_test_split(
    indices,
    test_size=0.25,
    random_state=SEMENTE,
    stratify=y
)

X_treino = X_texto.iloc[idx_treino]
X_teste = X_texto.iloc[idx_teste]
y_treino = y.iloc[idx_treino]
y_teste = y.iloc[idx_teste]

print('Registros de treino:', len(idx_treino))
print('Registros de teste:', len(idx_teste))

if EXECUTAR_TFIDF:
    tfidf_model = Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=1
        )),
        ('classificador', LogisticRegression(
            max_iter=3000,
            class_weight='balanced',
            random_state=SEMENTE
        ))
    ])

    tfidf_model.fit(X_treino, y_treino)
    pred_tfidf = tfidf_model.predict(X_teste)

    # Baseline: sempre prevê a classe mais frequente no conjunto de treino.
    baseline = DummyClassifier(strategy='most_frequent')
    baseline.fit(X_treino.to_frame(), y_treino)
    pred_baseline = baseline.predict(X_teste.to_frame())

    print('TF-IDF — relatório:')
    print(classification_report(
        y_teste, pred_tfidf,
        labels=ORDEM_CLASSES,
        zero_division=0,
        digits=4
    ))

    metricas_tfidf = pd.DataFrame([
        {
            'modelo': 'Dummy majoritário',
            'acuracia': accuracy_score(y_teste, pred_baseline),
            'acuracia_balanceada': balanced_accuracy_score(y_teste, pred_baseline),
            'f1_macro': f1_score(
                y_teste, pred_baseline, labels=ORDEM_CLASSES,
                average='macro', zero_division=0
            )
        },
        {
            'modelo': 'TF-IDF + regressão logística',
            'acuracia': accuracy_score(y_teste, pred_tfidf),
            'acuracia_balanceada': balanced_accuracy_score(y_teste, pred_tfidf),
            'f1_macro': f1_score(
                y_teste, pred_tfidf, labels=ORDEM_CLASSES,
                average='macro', zero_division=0
            )
        }
    ]).round(4)

    display(metricas_tfidf)

    ConfusionMatrixDisplay.from_predictions(
        y_teste,
        pred_tfidf,
        labels=ORDEM_CLASSES,
        display_labels=ORDEM_CLASSES,
        cmap='Blues',
        values_format='d'
    )
    plt.title('TF-IDF — reprodução dos rótulos da matriz')
    plt.tight_layout()
    plt.show()
else:
    print('TF-IDF desativado nas configurações.')

## 6. Experimento 2 — Sentence embeddings

Esta etapa converte textos em vetores semânticos. O classificador logístico usa a mesma divisão de treino/teste do TF-IDF. Em separado, calcula-se a similaridade de cosseno entre cada formação e as descrições dos domínios funcionais.

A similaridade é usada somente como sinal exploratório. Não existe um limiar universal que transforme similaridade em aderência; por isso, o notebook apresenta o domínio mais próximo, o segundo mais próximo e a diferença entre eles, sem declarar automaticamente uma classificação como “validada”.

In [ ]:
# ============================================================
# 10. SENTENCE EMBEDDINGS E SIMILARIDADE SEMÂNTICA
# ============================================================

embedder = None
embedding_model = None
emb_perfis = None
pred_emb = None

# Inicializa as colunas mesmo se a etapa for desativada ou falhar.
colunas_semanticas = [
    'DOMINIO_SEMANTICO_TOP',
    'SIMILARIDADE_TOP',
    'DOMINIO_SEMANTICO_SEGUNDO',
    'SIMILARIDADE_SEGUNDO',
    'DIFERENCA_SIMILARIDADE'
]
for coluna in colunas_semanticas:
    aplicaveis[coluna] = pd.NA

if EXECUTAR_EMBEDDINGS:
    try:
        from sentence_transformers import SentenceTransformer
        from sklearn.metrics.pairwise import cosine_similarity

        # Carrega o modelo multilíngue.
        embedder = SentenceTransformer(MODELO_EMBEDDING)

        # Gera embeddings dos mesmos textos usados no TF-IDF.
        # normalize_embeddings=True normaliza os vetores para facilitar
        # comparações por similaridade de cosseno.
        textos_todos = X_texto.tolist()
        X_emb = embedder.encode(
            textos_todos,
            normalize_embeddings=True,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        # Treina o classificador nos mesmos índices usados pelo TF-IDF.
        Xe_treino = X_emb[idx_treino]
        Xe_teste = X_emb[idx_teste]

        embedding_model = LogisticRegression(
            max_iter=3000,
            class_weight='balanced',
            random_state=SEMENTE
        )
        embedding_model.fit(Xe_treino, y_treino)

        pred_emb = embedding_model.predict(Xe_teste)

        print('Sentence embeddings — relatório:')
        print(classification_report(
            y_teste,
            pred_emb,
            labels=ORDEM_CLASSES,
            zero_division=0,
            digits=4
        ))

        metricas_embeddings = pd.DataFrame([{
            'modelo': 'Sentence embeddings + regressão logística',
            'acuracia': accuracy_score(y_teste, pred_emb),
            'acuracia_balanceada': balanced_accuracy_score(y_teste, pred_emb),
            'f1_macro': f1_score(
                y_teste, pred_emb, labels=ORDEM_CLASSES,
                average='macro', zero_division=0
            )
        }]).round(4)

        display(metricas_embeddings)

        ConfusionMatrixDisplay.from_predictions(
            y_teste,
            pred_emb,
            labels=ORDEM_CLASSES,
            display_labels=ORDEM_CLASSES,
            cmap='Greens',
            values_format='d'
        )
        plt.title('Sentence embeddings — reprodução dos rótulos da matriz')
        plt.tight_layout()
        plt.show()

        # Descrições funcionais para comparar com as formações acadêmicas.
        dominios_perfil = list(HIGH.keys())

        # Usa descrições funcionais consistentes com a função de perfil.
        textos_perfis = [perfil_funcional(d) for d in dominios_perfil]
        emb_perfis = embedder.encode(
            textos_perfis,
            normalize_embeddings=True,
            convert_to_numpy=True
        )

        # Calcula a similaridade usando somente a formação acadêmica,
        # sem incluir o texto da unidade nessa comparação.
        textos_formacao = (
            aplicaveis['AREA_FORMACAO_LIMPA']
            .fillna('')
            .astype(str)
            .tolist()
        )

        emb_formacoes = embedder.encode(
            textos_formacao,
            normalize_embeddings=True,
            show_progress_bar=True,
            convert_to_numpy=True
        )

        matriz_sim = cosine_similarity(emb_formacoes, emb_perfis)

        # Seleciona as duas maiores similaridades por registro.
        ordem_indices = np.argsort(matriz_sim, axis=1)[:, ::-1]
        idx_top1 = ordem_indices[:, 0]
        idx_top2 = ordem_indices[:, 1]

        sim_top1 = matriz_sim[np.arange(len(matriz_sim)), idx_top1]
        sim_top2 = matriz_sim[np.arange(len(matriz_sim)), idx_top2]

        aplicaveis['DOMINIO_SEMANTICO_TOP'] = [
            dominios_perfil[i] for i in idx_top1
        ]
        aplicaveis['SIMILARIDADE_TOP'] = np.round(sim_top1, 4)

        aplicaveis['DOMINIO_SEMANTICO_SEGUNDO'] = [
            dominios_perfil[i] for i in idx_top2
        ]
        aplicaveis['SIMILARIDADE_SEGUNDO'] = np.round(sim_top2, 4)

        aplicaveis['DIFERENCA_SIMILARIDADE'] = np.round(
            sim_top1 - sim_top2, 4
        )

        print(
            'A similaridade é uma medida de proximidade vetorial, '
            'não uma porcentagem de aderência.'
        )

    except Exception as erro:
        warnings.warn(
            'A etapa de embeddings não foi concluída. '
            f'Detalhe técnico: {repr(erro)}'
        )
        print(
            'TF-IDF e a matriz de regras continuam disponíveis. '
            'Verifique conexão, instalação e memória para executar embeddings.'
        )
else:
    print('Sentence embeddings desativados nas configurações.')

# Exibe uma amostra sem identificadores diretos.
colunas_exibicao = [
    'ID_REGISTRO_ANALISE',
    'AREA_FORMACAO',
    'UNIDADE',
    'TIPO_UNIDADE_OFICIAL',
    'DOMINIO_UNIDADE',
    'ADERENCIA_CATEGORIA',
    'DOMINIO_SEMANTICO_TOP',
    'SIMILARIDADE_TOP',
    'DOMINIO_SEMANTICO_SEGUNDO',
    'SIMILARIDADE_SEGUNDO',
    'DIFERENCA_SIMILARIDADE'
]
display(aplicaveis[colunas_exibicao].head(25))

## 7. Comparação entre os modelos e interpretação

A tabela de métricas só é comparável se os modelos tiverem usado os mesmos registros de treino e teste. O F1-macro dá o mesmo peso a cada classe, sendo útil quando as classes estão desbalanceadas.

Mesmo que um modelo apresente desempenho elevado, isso pode ocorrer porque os rótulos foram produzidos por regras determinísticas com base nos mesmos campos textuais. Para validar a matriz, é necessário um conjunto de referência rotulado independentemente por especialistas, preferencialmente sem que conheçam a classificação automática.

In [ ]:
# ============================================================
# 11. QUADRO COMPARATIVO DOS EXPERIMENTOS
# ============================================================

linhas_metricas = []

if EXECUTAR_TFIDF and pred_tfidf is not None:
    linhas_metricas.append({
        'modelo': 'TF-IDF + regressão logística',
        'acuracia': accuracy_score(y_teste, pred_tfidf),
        'acuracia_balanceada': balanced_accuracy_score(y_teste, pred_tfidf),
        'f1_macro': f1_score(
            y_teste, pred_tfidf, labels=ORDEM_CLASSES,
            average='macro', zero_division=0
        )
    })

if EXECUTAR_EMBEDDINGS and pred_emb is not None:
    linhas_metricas.append({
        'modelo': 'Sentence embeddings + regressão logística',
        'acuracia': accuracy_score(y_teste, pred_emb),
        'acuracia_balanceada': balanced_accuracy_score(y_teste, pred_emb),
        'f1_macro': f1_score(
            y_teste, pred_emb, labels=ORDEM_CLASSES,
            average='macro', zero_division=0
        )
    })

if linhas_metricas:
    comparacao_modelos = pd.DataFrame(linhas_metricas).round(4)
    display(comparacao_modelos)
else:
    print('Nenhum modelo supervisionado foi executado.')

## 8. Exportação dos resultados

São exportadas duas tabelas:
- base analítica completa, com códigos e categorias;
- subconjunto aplicável à análise de aderência.

Colunas identificadoras diretas removidas na etapa de preparação não são reintroduzidas. O identificador sequencial serve somente para rastreabilidade dentro desta execução; não deve ser tratado como anonimização formal.

In [ ]:
# ============================================================
# 12. EXPORTAÇÃO DOS RESULTADOS
# ============================================================

# Arquivo de saída sem nomes, CPF, matrícula ou outros identificadores
# diretos removidos na preparação.
ARQUIVO_SAIDA_COMPLETO = 'resultado_analise_pmac_completo.csv'
ARQUIVO_SAIDA_APLICAVEL = 'resultado_analise_pmac_aplicavel.csv'
ARQUIVO_SAIDA_AUDITORIA = 'auditoria_classificacao_unidades.csv'

work.to_csv(
    ARQUIVO_SAIDA_COMPLETO,
    index=False,
    encoding='utf-8-sig'
)

aplicaveis.to_csv(
    ARQUIVO_SAIDA_APLICAVEL,
    index=False,
    encoding='utf-8-sig'
)

auditoria_unidades.to_csv(
    ARQUIVO_SAIDA_AUDITORIA,
    index=False,
    encoding='utf-8-sig'
)

print('Arquivos gerados:')
print(ARQUIVO_SAIDA_COMPLETO)
print(ARQUIVO_SAIDA_APLICAVEL)
print(ARQUIVO_SAIDA_AUDITORIA)

# No Google Colab, descomente as linhas abaixo para baixar os arquivos.
# files.download(ARQUIVO_SAIDA_COMPLETO)
# files.download(ARQUIVO_SAIDA_APLICAVEL)
# files.download(ARQUIVO_SAIDA_AUDITORIA)